# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import torch
import gc
import pickle
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer
from sklearn.metrics import classification_report
from torch.utils.data import Dataset
from IPython.display import display

c:\Users\VICTUS\OneDrive\Documents\Semester_4\ML\ML\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Preparation

In [2]:
# 1. PATH SETUP
path_test_data = '../data/data_test_master.csv'
path_base_model = '../models/bert/'
path_model_optuna = '../models/distilbert_optuna/best_model/'

# 2. LOAD TEST DATA
test_df = pd.read_csv(path_test_data).dropna(subset=['clean_text'])

X_test_text = test_df['clean_text'].astype(str).tolist()
y_test_text = test_df['emotion']

# 3. LOAD LABEL ENCODER
with open(f"{path_model_optuna}label_encoder.pkl", 'rb') as f:
    label_encoder = pickle.load(f)

y_test_encoded = label_encoder.transform(y_test_text)

# 4. TOKENIZER & DATASET
tokenizer = DistilBertTokenizer.from_pretrained(path_base_model)

test_encodings = tokenizer(
    X_test_text,
    truncation=True,
    padding=True,
    max_length=60
)

class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

test_eval_dataset = EmotionDataset(test_encodings, y_test_encoded)

# Prediction using both models

In [3]:
# 5. BASELINE DISTILBERT PREDICTION
base_model = DistilBertForSequenceClassification.from_pretrained(path_base_model)
base_trainer = Trainer(model=base_model)

preds_lama = base_trainer.predict(test_eval_dataset)

y_pred_baseline = label_encoder.inverse_transform(
    np.argmax(preds_lama.predictions, axis=1)
)

# 6. OPTUNA-TUNED DISTILBERT PREDICTION
model_optuna = DistilBertForSequenceClassification.from_pretrained(path_model_optuna)
trainer_optuna = Trainer(model=model_optuna)

preds_optuna = trainer_optuna.predict(test_eval_dataset)

y_pred_optuna = label_encoder.inverse_transform(
    np.argmax(preds_optuna.predictions, axis=1)
)

100%|██████████| 957/957 [00:06<00:00, 141.73it/s]


# Evaluation on Both Models

In [4]:
# 7. PERFORMANCE COMPARISON REPORT
report_base = classification_report(
    y_test_text,
    y_pred_baseline,
    target_names=label_encoder.classes_,
    output_dict=True
)

report_optuna = classification_report(
    y_test_text,
    y_pred_optuna,
    target_names=label_encoder.classes_,
    output_dict=True
)

metrics = ['precision', 'recall', 'f1-score']
emotions = label_encoder.classes_.tolist() + [
    'accuracy',
    'macro avg',
    'weighted avg'
]

data = []

for emo in emotions:
    if emo == 'accuracy':
        row = [
            report_base[emo], report_optuna[emo], report_base[emo] - report_optuna[emo],
            report_base[emo], report_optuna[emo], report_base[emo] - report_optuna[emo],
            report_base[emo], report_optuna[emo], report_base[emo] - report_optuna[emo]
        ]
    else:
        row = [
            report_base[emo]['precision'], report_optuna[emo]['precision'], report_base[emo]['precision'] - report_optuna[emo]['precision'],
            report_base[emo]['recall'], report_optuna[emo]['recall'], report_base[emo]['recall'] - report_optuna[emo]['recall'],
            report_base[emo]['f1-score'], report_optuna[emo]['f1-score'], report_base[emo]['f1-score'] - report_optuna[emo]['f1-score']
        ]
    data.append(row)

columns = pd.MultiIndex.from_tuples([
    ('Precision', 'Baseline'), ('Precision', 'Optuna'), ('Precision', 'Difference'),
    ('Recall', 'Baseline'), ('Recall', 'Optuna'), ('Recall', 'Difference'),
    ('F1-Score', 'Baseline'), ('F1-Score', 'Optuna'), ('F1-Score', 'Difference')
])

In [5]:
df = pd.DataFrame(data, index=emotions, columns=columns)

print("\n=== COMPLETE PERFORMANCE COMPARISON ===")
print("Note: Positive difference means the Baseline model performs better, while negative difference means the Optuna-Tuned model performs better.")

display(df.round(2))


=== COMPLETE PERFORMANCE COMPARISON ===
Note: Positive difference means the Baseline model performs better, while negative difference means the Optuna-Tuned model performs better.


Precision                     Recall                   F1-Score  \
              Baseline Optuna Difference Baseline Optuna Difference Baseline   
Anger             0.76   0.76       0.00     0.79   0.77       0.02     0.78   
Fear              0.67   0.67       0.00     0.73   0.71       0.02     0.70   
Joy               0.80   0.80       0.01     0.72   0.71       0.01     0.76   
Love              0.83   0.82       0.01     0.89   0.88       0.01     0.86   
Sadness           0.76   0.73       0.03     0.71   0.72      -0.02     0.73   
Surprise          0.70   0.70      -0.00     0.72   0.72       0.00     0.71   
accuracy          0.77   0.76       0.01     0.77   0.76       0.01     0.77   
macro avg         0.75   0.75       0.01     0.76   0.75       0.01     0.76   
weighted avg      0.77   0.76       0.01     0.77   0.76       0.01     0.77   

                                
             Optuna Difference  
Anger          0.77       0.01  
Fear           0.69       0.01  
Joy            0.75       0.01  
Love           0.85       0.01  
Sadness        0.73       0.00  
Surprise       0.71      -0.00  
accuracy       0.76       0.01  
macro avg      0.75       0.01  
weighted avg   0.76       0.01

As shown in the comparison results, the performance difference between the baseline model and the Optuna-tuned model is relatively insignificant. In several metrics, the baseline model even performs slightly better than the optimized version. Therefore, for future development and deployment, we decided to continue using the baseline DistilBERT model.